# NAS Competition – lokaler Upload in Google Colab

Dieses Notebook richtet das Starter-Kit neu ein, lädt Datensätze und `submission.zip` direkt vom lokalen PC hoch, prüft ihre Struktur und startet die Auswertung.

Vor dem Start: **Laufzeit → Laufzeittyp ändern → T4 GPU** auswählen.

Die Datensätze sollten als ZIP hochgeladen werden. Beide Strukturen werden erkannt:

```text
datasets.zip
├── AddNIST/
│   ├── metadata
│   ├── train_x.npy
│   ├── train_y.npy
│   ├── valid_x.npy
│   ├── valid_y.npy
│   ├── test_x.npy
│   └── test_y.npy
└── Gutenberg/...
```

Ein zusätzlicher oberster Ordner wie `datasets/` ist ebenfalls erlaubt. Führe die Zellen der Reihe nach aus. Upload-Zellen müssen in der aktuellen Browser-Sitzung neu ausgeführt werden; ein gespeichertes Upload-Widget ist nicht wiederverwendbar.

In [ ]:
# 1. Frisches Starter-Kit einrichten
from pathlib import Path
import shutil
import subprocess

ROOT = Path('/content/NAS-Comp')
REPO_URL = 'https://github.com/Towers-D/NAS-Comp-Starter-Kit.git'

if ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)], check=True)
print(f'Starter-Kit bereit: {ROOT}')

## Datensätze vom lokalen PC hochladen

Wähle eine oder mehrere ZIP-Dateien aus. Das Notebook sucht darin selbstständig nach gültigen Dataset-Ordnern. Einzelne `.npy`-Dateien werden absichtlich nicht akzeptiert, weil der Browser-Upload ihre Ordnerstruktur nicht zuverlässig überträgt.

In [ ]:
# 2. Dataset-ZIP(s) hochladen, sicher entpacken und validieren
from google.colab import files
from pathlib import Path
import json
import shutil
import tempfile
import zipfile
import numpy as np

REQUIRED_DATASET_FILES = {
    'metadata', 'train_x.npy', 'train_y.npy',
    'valid_x.npy', 'valid_y.npy', 'test_x.npy', 'test_y.npy'
}

def safe_extract(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise ValueError(f'Unsicherer Pfad in ZIP: {member.filename}')
        archive.extractall(destination)

print('Bitte Dataset-ZIP(s) auswählen:')
uploaded_datasets = files.upload()
if not uploaded_datasets:
    raise RuntimeError('Es wurde keine Datei hochgeladen.')

work_dir = Path(tempfile.mkdtemp(prefix='nas_datasets_', dir='/content'))
try:
    for uploaded_name in uploaded_datasets:
        archive_path = Path.cwd() / uploaded_name
        if archive_path.suffix.lower() != '.zip':
            raise ValueError(f'{uploaded_name} ist keine ZIP-Datei.')
        extract_dir = work_dir / archive_path.stem
        extract_dir.mkdir(parents=True, exist_ok=True)
        safe_extract(archive_path, extract_dir)

    candidates = []
    for metadata_file in work_dir.rglob('metadata'):
        folder = metadata_file.parent
        present = {item.name for item in folder.iterdir() if item.is_file()}
        if REQUIRED_DATASET_FILES <= present:
            candidates.append(folder)

    unique_candidates = {folder.resolve(): folder for folder in candidates}
    candidates = sorted(unique_candidates.values(), key=lambda p: str(p))
    if not candidates:
        raise ValueError(
            'Kein gültiger Dataset-Ordner gefunden. Erwartet werden metadata sowie '
            'train/valid/test_x.npy und train/valid/test_y.npy.'
        )

    datasets_dir = ROOT / 'datasets'
    if datasets_dir.exists():
        shutil.rmtree(datasets_dir)
    datasets_dir.mkdir(parents=True)

    seen_names = set()
    summaries = []
    for source in candidates:
        if source.name in seen_names:
            raise ValueError(f'Doppelter Dataset-Ordnername: {source.name}')
        seen_names.add(source.name)

        with open(source / 'metadata', encoding='utf-8') as handle:
            metadata = json.load(handle)
        for key in ('num_classes', 'input_shape', 'codename', 'benchmark'):
            if key not in metadata:
                raise ValueError(f'{source.name}/metadata: Schlüssel {key!r} fehlt.')

        train_x = np.load(source / 'train_x.npy', mmap_mode='r')
        train_y = np.load(source / 'train_y.npy', mmap_mode='r')
        valid_x = np.load(source / 'valid_x.npy', mmap_mode='r')
        valid_y = np.load(source / 'valid_y.npy', mmap_mode='r')
        test_x = np.load(source / 'test_x.npy', mmap_mode='r')
        test_y = np.load(source / 'test_y.npy', mmap_mode='r')
        if len(train_x) != len(train_y) or len(valid_x) != len(valid_y) or len(test_x) != len(test_y):
            raise ValueError(f'{source.name}: Anzahl Bilder und Labels stimmt nicht überein.')

        shutil.copytree(source, datasets_dir / source.name)
        summaries.append({
            'Ordner': source.name,
            'Codename': metadata['codename'],
            'Train': tuple(train_x.shape),
            'Valid': tuple(valid_x.shape),
            'Test': tuple(test_x.shape),
            'Klassen': metadata['num_classes'],
            'Zeitlimit (h)': metadata.get('time_limit', 0.5),
        })

    print(f'\n{len(summaries)} Dataset(s) erfolgreich installiert:')
    for summary in summaries:
        print(summary)
finally:
    shutil.rmtree(work_dir, ignore_errors=True)

## Submission vom lokalen PC hochladen

Wähle deine `submission.zip`. Der tatsächliche Browser-Dateiname ist egal; auch Namen wie `submission (1).zip` werden korrekt verarbeitet. Sowohl Dateien direkt im ZIP als auch ein umschließender `submission/`-Ordner werden erkannt.

In [ ]:
# 3. Submission hochladen, entpacken und prüfen
from google.colab import files
from pathlib import Path
import shutil
import tempfile

print('Bitte submission.zip auswählen:')
uploaded_submission = files.upload()
zip_names = [name for name in uploaded_submission if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError(f'Bitte genau eine ZIP-Datei auswählen; gefunden: {zip_names}')

submission_work = Path(tempfile.mkdtemp(prefix='nas_submission_', dir='/content'))
try:
    safe_extract(Path.cwd() / zip_names[0], submission_work)
    required_code = {'data_processor.py', 'nas.py', 'trainer.py'}
    code_roots = []
    for nas_file in submission_work.rglob('nas.py'):
        folder = nas_file.parent
        present = {item.name for item in folder.iterdir() if item.is_file()}
        if required_code <= present:
            code_roots.append(folder)
    if len(code_roots) != 1:
        raise ValueError(
            'Die ZIP muss genau einen Ordner mit data_processor.py, nas.py und trainer.py enthalten. '
            f'Gefundene Kandidaten: {[str(p) for p in code_roots]}'
        )

    submission_dir = ROOT / 'submission'
    if submission_dir.exists():
        shutil.rmtree(submission_dir)
    shutil.copytree(code_roots[0], submission_dir)
    installed = sorted(p.name for p in submission_dir.iterdir())
    print('Submission erfolgreich installiert:')
    print('\n'.join(f'  - {name}' for name in installed))
finally:
    shutil.rmtree(submission_work, ignore_errors=True)

In [ ]:
# 4. Umgebung, Python-Dateien und bekannte Starter-Kit-Inkonsistenz prüfen
import re
import subprocess
import sys
import torch

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA verfügbar: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNUNG: Keine GPU aktiv. In Colab Laufzeit → Laufzeittyp ändern wählen.')

# Manche Commits des Starter-Kits verlangen grace_time, rufen die Funktion aber
# mit nur zwei Argumenten auf. Nur diese lokale Testkopie erhält dann einen Default.
evaluation_main = ROOT / 'evaluation' / 'main.py'
source = evaluation_main.read_text(encoding='utf-8')
patched = re.sub(
    r'grace_time\s*:\s*bool\s*\)',
    'grace_time: bool=False)',
    source,
)
if patched != source:
    evaluation_main.write_text(patched, encoding='utf-8')
    print('Bekannte is_out_of_time-Inkonsistenz in der lokalen Testkopie behoben.')
else:
    print('Starter-Kit-Zeitprüfung ist bereits konsistent.')

python_files = sorted(str(path) for path in (ROOT / 'submission').glob('*.py'))
subprocess.run([sys.executable, '-m', 'py_compile', *python_files], check=True)
print(f'Syntaxprüfung bestanden ({len(python_files)} Python-Dateien).')

## Vollständige lokale Bewertung starten

Diese Zelle kann abhängig von Anzahl und `time_limit` der Datensätze lange laufen. Die Ausgabe erscheint fortlaufend unter der Zelle.

In [ ]:
# 5. Build → NAS/Training/Prediction → Scoring
%cd /content/NAS-Comp
!make submission=submission all

In [ ]:
# Optional: Vorhersagen und Statistiken als ZIP herunterladen
from google.colab import files
from pathlib import Path
import shutil

predictions_dir = ROOT / 'package' / 'predictions'
if not predictions_dir.exists():
    raise FileNotFoundError('Noch keine Vorhersagen gefunden. Zuerst die Bewertungszelle ausführen.')

archive = shutil.make_archive('/content/nas_predictions', 'zip', predictions_dir)
files.download(archive)